# Wire Offset by Target Area

This notebook demonstrates creating offset wires to achieve a target area using **topologic_fast**.

## Concept

The goal is to create an inner wire (hole) from an outer wire boundary such that the inner face
has a specific target area. This is useful in architectural applications where you need to:
- Create window openings with specific glazing areas
- Design floor plans with target room sizes
- Generate frames with precise material requirements

## Note on topologicpy vs topologic_fast

The original topologicpy implementation uses `Wire.ByOffsetArea()` with scipy optimization
to iteratively find the correct offset. In topologic_fast, we demonstrate a simpler approach
using binary search to find the uniform offset that achieves the target area.

In [ ]:
import topologic_fast as tf
import plotly.graph_objects as go
import math

## Helper Functions for Visualization

First, let's create helper functions to visualize wires and faces using Plotly.

In [ ]:
def wire_to_plotly_trace(wire, color='blue', width=2, name='Wire', fill=False, fillcolor='lightblue'):
    """Convert a wire to a Plotly scatter trace for 2D visualization."""
    vertices = wire.Vertices()
    coords = [v.Coordinates() for v in vertices]
    
    # Close the wire for visualization
    if wire.IsClosed() and len(coords) > 0:
        coords.append(coords[0])
    
    x = [c[0] for c in coords]
    y = [c[1] for c in coords]
    
    if fill:
        return go.Scatter(
            x=x, y=y,
            mode='lines',
            fill='toself',
            fillcolor=fillcolor,
            line=dict(color=color, width=width),
            name=name
        )
    else:
        return go.Scatter(
            x=x, y=y,
            mode='lines',
            line=dict(color=color, width=width),
            name=name
        )


def face_to_plotly_trace(face, color='lightblue', edge_color='black', name='Face'):
    """Convert a face to a Plotly scatter trace for 2D visualization."""
    wire = face.ExternalBoundary()
    return wire_to_plotly_trace(wire, color=edge_color, fill=True, fillcolor=color, name=name)


def show_wires_2d(wires, colors=None, names=None, title='Wire Visualization'):
    """Display multiple wires in a 2D plot."""
    fig = go.Figure()
    
    if colors is None:
        colors = ['blue', 'red', 'green', 'orange', 'purple'] * len(wires)
    if names is None:
        names = [f'Wire {i+1}' for i in range(len(wires))]
    
    for wire, color, name in zip(wires, colors, names):
        trace = wire_to_plotly_trace(wire, color=color, name=name)
        fig.add_trace(trace)
    
    fig.update_layout(
        title=title,
        xaxis=dict(title='X', scaleanchor='y', scaleratio=1),
        yaxis=dict(title='Y'),
        width=700,
        height=600,
        showlegend=True
    )
    
    return fig

## Create a Base Wire

Let's create a rectangular wire as our outer boundary.

In [ ]:
# Create a rectangular wire (4x6 units)
outer_wire = tf.Wire.Rectangle(width=4.0, length=6.0)

# Calculate the outer area
outer_face = tf.Face.ByWire(outer_wire)
outer_area = outer_face.Area()

print(f"Outer Wire Properties:")
print(f"  Closed: {outer_wire.IsClosed()}")
print(f"  Length (perimeter): {outer_wire.Length():.2f}")
print(f"  Area: {outer_area:.2f} square units")
print(f"  Vertices: {len(outer_wire.Vertices())}")
print(f"  Edges: {len(outer_wire.Edges())}")

In [ ]:
# Visualize the outer wire
fig = show_wires_2d([outer_wire], colors=['blue'], names=['Outer Boundary'], 
                   title='Original Rectangle (4x6)')
fig.show()

## Offset Wire to Target Area

Now, let's create a function that computes an offset wire to achieve a target inner area.
We use binary search to find the uniform offset distance.

**Note:** The topologicpy `Wire.ByOffsetArea()` function supports per-edge offset constraints
using scipy optimization. This is not yet available in topologic_fast, so we demonstrate
a uniform offset approach.

In [ ]:
def wire_by_uniform_offset(wire, offset):
    """
    Create an offset wire with a uniform offset distance.
    
    For a rectangle, we can compute this analytically by creating a smaller rectangle.
    For general polygons, we would need to offset each edge and compute intersections.
    
    Parameters:
    -----------
    wire : tf.Wire
        The input wire (must be a closed, planar wire)
    offset : float
        The offset distance (positive = inward for outer boundary)
    
    Returns:
    --------
    tf.Wire
        The offset wire
    """
    # Get the bounding box to determine dimensions
    bbox = wire.BoundingBox()
    min_pt, max_pt = bbox
    
    width = max_pt[0] - min_pt[0]
    length = max_pt[1] - min_pt[1]
    
    # Compute the center
    center_x = (min_pt[0] + max_pt[0]) / 2
    center_y = (min_pt[1] + max_pt[1]) / 2
    
    # Create a smaller rectangle offset inward
    new_width = width - 2 * offset
    new_length = length - 2 * offset
    
    if new_width <= 0 or new_length <= 0:
        return None  # Offset too large
    
    # Create vertices for the new rectangle
    half_w = new_width / 2
    half_l = new_length / 2
    
    v1 = tf.Vertex.ByCoordinates(center_x - half_w, center_y - half_l, 0)
    v2 = tf.Vertex.ByCoordinates(center_x + half_w, center_y - half_l, 0)
    v3 = tf.Vertex.ByCoordinates(center_x + half_w, center_y + half_l, 0)
    v4 = tf.Vertex.ByCoordinates(center_x - half_w, center_y + half_l, 0)
    
    return tf.Wire.ByVertices([v1, v2, v3, v4], close=True)


def wire_by_target_area(outer_wire, target_area, max_iterations=50, tolerance=0.01):
    """
    Create an inner wire (hole) such that the inner face has the target area.
    
    Uses binary search to find the correct uniform offset.
    
    Parameters:
    -----------
    outer_wire : tf.Wire
        The outer boundary wire
    target_area : float
        The desired area of the inner (hole) region
    max_iterations : int
        Maximum number of binary search iterations
    tolerance : float
        Acceptable area tolerance
    
    Returns:
    --------
    tuple
        (inner_wire, actual_area, offset_used)
    """
    # Get outer dimensions for maximum offset
    bbox = outer_wire.BoundingBox()
    min_pt, max_pt = bbox
    width = max_pt[0] - min_pt[0]
    length = max_pt[1] - min_pt[1]
    
    # Binary search bounds
    min_offset = 0.0
    max_offset = min(width, length) / 2 - 0.001  # Leave at least a tiny area
    
    outer_face = tf.Face.ByWire(outer_wire)
    outer_area = outer_face.Area()
    
    if target_area >= outer_area:
        print(f"Warning: Target area ({target_area}) >= outer area ({outer_area})")
        return outer_wire, outer_area, 0.0
    
    best_wire = None
    best_area = 0
    best_offset = 0
    
    for i in range(max_iterations):
        mid_offset = (min_offset + max_offset) / 2
        
        inner_wire = wire_by_uniform_offset(outer_wire, mid_offset)
        if inner_wire is None:
            max_offset = mid_offset
            continue
        
        inner_face = tf.Face.ByWire(inner_wire)
        inner_area = inner_face.Area()
        
        best_wire = inner_wire
        best_area = inner_area
        best_offset = mid_offset
        
        if abs(inner_area - target_area) < tolerance:
            print(f"Converged in {i+1} iterations")
            break
        
        if inner_area > target_area:
            # Need smaller area, increase offset
            min_offset = mid_offset
        else:
            # Need larger area, decrease offset
            max_offset = mid_offset
    
    return best_wire, best_area, best_offset

## Example 1: Target Area of 12 Square Units

Starting with a 4x6 rectangle (24 sq units), create a hole with area of 12 sq units.

In [ ]:
target_area = 12.0

inner_wire, actual_area, offset = wire_by_target_area(outer_wire, target_area)

print(f"\nResults:")
print(f"  Target Area: {target_area:.2f} sq units")
print(f"  Actual Area: {actual_area:.2f} sq units")
print(f"  Offset Used: {offset:.4f} units")
print(f"  Error: {abs(actual_area - target_area):.4f} sq units ({abs(actual_area - target_area)/target_area*100:.2f}%)")

In [ ]:
# Visualize outer and inner wires
fig = go.Figure()

# Add outer wire with fill
fig.add_trace(wire_to_plotly_trace(outer_wire, color='blue', name='Outer Boundary',
                                   fill=True, fillcolor='lightblue'))

# Add inner wire (hole) with different fill
fig.add_trace(wire_to_plotly_trace(inner_wire, color='red', name=f'Inner Hole (Area={actual_area:.2f})',
                                   fill=True, fillcolor='white'))

fig.update_layout(
    title=f'Wire Offset by Target Area (Target: {target_area}, Actual: {actual_area:.2f})',
    xaxis=dict(title='X', scaleanchor='y', scaleratio=1),
    yaxis=dict(title='Y'),
    width=700,
    height=600,
    showlegend=True
)

fig.show()

## Example 2: Create Face with Hole

Now let's create a face with the inner wire as an internal boundary (hole).

In [ ]:
# Create face with hole
face_with_hole = tf.Face.ByExternalInternalBoundaries(outer_wire, [inner_wire])

print(f"Face with Hole Properties:")
print(f"  Total Area (outer - inner): {face_with_hole.Area():.2f} sq units")
print(f"  Perimeter: {face_with_hole.Perimeter():.2f} units")
print(f"  External Boundary: {face_with_hole.ExternalBoundary()}")
print(f"  Internal Boundaries: {len(face_with_hole.InternalBoundaries())} hole(s)")

## Example 3: Multiple Target Areas

Let's visualize how different target areas produce different offsets.

In [ ]:
# Create multiple inner wires for different target areas
target_areas = [18.0, 14.0, 10.0, 6.0, 2.0]
colors = ['red', 'orange', 'green', 'purple', 'brown']

fig = go.Figure()

# Add outer wire
fig.add_trace(wire_to_plotly_trace(outer_wire, color='blue', width=3, name='Outer (24 sq units)'))

# Add inner wires for each target area
for target, color in zip(target_areas, colors):
    inner_w, actual_a, off = wire_by_target_area(outer_wire, target, tolerance=0.001)
    fig.add_trace(wire_to_plotly_trace(inner_w, color=color, width=2,
                                       name=f'Target={target} (Actual={actual_a:.1f}, Offset={off:.3f})'))

fig.update_layout(
    title='Wire Offsets for Different Target Areas',
    xaxis=dict(title='X', scaleanchor='y', scaleratio=1, range=[-3, 3]),
    yaxis=dict(title='Y', range=[-4, 4]),
    width=800,
    height=700,
    showlegend=True,
    legend=dict(x=1.05, y=1)
)

fig.show()

## Example 4: Different Wire Shapes

Let's try with a circular wire.

In [ ]:
def circle_wire_by_target_area(outer_radius, target_area, sides=32):
    """
    Create an inner circular wire with target area.
    
    For a circle, Area = pi * r^2, so r = sqrt(Area / pi)
    """
    # Create outer circle
    outer_wire = tf.Wire.Circle(radius=outer_radius, sides=sides)
    outer_area = math.pi * outer_radius**2
    
    # Calculate inner radius for target area
    inner_radius = math.sqrt(target_area / math.pi)
    
    if inner_radius >= outer_radius:
        print(f"Warning: Target area too large for circle")
        return outer_wire, outer_wire, outer_area
    
    inner_wire = tf.Wire.Circle(radius=inner_radius, sides=sides)
    actual_area = math.pi * inner_radius**2
    
    return outer_wire, inner_wire, actual_area


# Create circular wires
outer_r = 3.0
target_circle_area = 10.0

circle_outer, circle_inner, circle_actual = circle_wire_by_target_area(outer_r, target_circle_area)

outer_circle_area = math.pi * outer_r**2
inner_radius_calc = math.sqrt(target_circle_area / math.pi)

print(f"Circular Wire Results:")
print(f"  Outer Radius: {outer_r:.2f}")
print(f"  Outer Area: {outer_circle_area:.2f} sq units")
print(f"  Target Inner Area: {target_circle_area:.2f} sq units")
print(f"  Calculated Inner Radius: {inner_radius_calc:.4f}")
print(f"  Actual Inner Area: {circle_actual:.2f} sq units")

In [ ]:
# Visualize circular wires
fig = go.Figure()

fig.add_trace(wire_to_plotly_trace(circle_outer, color='blue', width=2, 
                                   name=f'Outer Circle (r={outer_r})', 
                                   fill=True, fillcolor='lightblue'))
fig.add_trace(wire_to_plotly_trace(circle_inner, color='red', width=2,
                                   name=f'Inner Circle (Area={circle_actual:.2f})',
                                   fill=True, fillcolor='white'))

fig.update_layout(
    title=f'Circular Wire with Target Hole Area = {target_circle_area}',
    xaxis=dict(title='X', scaleanchor='y', scaleratio=1, range=[-4, 4]),
    yaxis=dict(title='Y', range=[-4, 4]),
    width=600,
    height=600,
    showlegend=True
)

fig.show()

## Example 5: Star Shape

Let's try with a more complex star shape.

In [ ]:
# Create a star wire
star_outer = tf.Wire.Star(radius_a=3.0, radius_b=1.5, rays=6)
star_face = tf.Face.ByWire(star_outer)
star_area = star_face.Area()

print(f"Star Wire Properties:")
print(f"  Outer Radius: 3.0")
print(f"  Inner Radius: 1.5")
print(f"  Rays: 6")
print(f"  Area: {star_area:.2f} sq units")
print(f"  Perimeter: {star_outer.Length():.2f} units")

In [ ]:
# Create a scaled-down version of the star as a "hole"
# NOTE: topologic_fast currently does not have Wire.ByOffset for arbitrary shapes
# We can create a smaller star by scaling the radii

scale_factor = 0.6
star_inner = tf.Wire.Star(radius_a=3.0*scale_factor, radius_b=1.5*scale_factor, rays=6)
star_inner_face = tf.Face.ByWire(star_inner)
star_inner_area = star_inner_face.Area()

print(f"\nInner Star (scaled by {scale_factor}):")
print(f"  Area: {star_inner_area:.2f} sq units")
print(f"  Frame Area: {star_area - star_inner_area:.2f} sq units")

In [ ]:
# Visualize star wires
fig = go.Figure()

fig.add_trace(wire_to_plotly_trace(star_outer, color='blue', width=2, 
                                   name=f'Outer Star (Area={star_area:.2f})', 
                                   fill=True, fillcolor='gold'))
fig.add_trace(wire_to_plotly_trace(star_inner, color='red', width=2,
                                   name=f'Inner Star (Area={star_inner_area:.2f})',
                                   fill=True, fillcolor='white'))

fig.update_layout(
    title='Star Wire with Scaled Inner Hole',
    xaxis=dict(title='X', scaleanchor='y', scaleratio=1, range=[-4, 4]),
    yaxis=dict(title='Y', range=[-4, 4]),
    width=600,
    height=600,
    showlegend=True
)

fig.show()

## Summary

This notebook demonstrated:

1. **Wire Creation**: Using `tf.Wire.Rectangle()`, `tf.Wire.Circle()`, and `tf.Wire.Star()` to create various closed wires

2. **Area Calculation**: Using `tf.Face.ByWire()` to create faces and calculate their areas

3. **Offset by Target Area**: A binary search algorithm to find the offset that produces a target inner area

4. **Faces with Holes**: Using `tf.Face.ByExternalInternalBoundaries()` to create faces with internal boundaries

### API Differences from topologicpy

| topologicpy | topologic_fast | Notes |
|------------|----------------|-------|
| `Wire.ByOffsetArea()` | Not available | Use binary search with uniform offset |
| `Wire.ByOffset()` | Not available | Manual offset calculation needed |
| `Wire.Einstein()` | Not available | Special shapes not yet implemented |
| `Dictionary.SetValueAtKey()` | `tf.Dictionary.SetValueAtKey()` | Similar API |

### Applications

- Window glazing with specific glass area requirements
- Frame design with material constraints
- Floor plan optimization for room sizes
- HVAC duct sizing